In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np

# ── Data ──────────────────────────────────────────────────────────────────────
df = pd.read_excel(
    '/mnt/user-data/uploads/1781788381570_June_to_Nov_actual_sales.xlsx',
    sheet_name='MoM_Sales'
)

VALID_MONTHS = ['June', 'July', 'August', 'September', 'October', 'November']

colors = {
    'FY_Total'  : '#1f4e79',
    'June'      : '#2e86ab',
    'July'      : '#e84855',
    'August'    : '#f18f01',
    'September' : '#5c4b8a',
    'October'   : '#3bb273',
    'November'  : '#c94040',
}

markers = {
    'FY_Total'  : 'D',
    'June'      : 'o',
    'July'      : 's',
    'August'    : '^',
    'September' : 'P',
    'October'   : 'X',
    'November'  : 'v',
}

def to_lakhs(val):
    return f"{val / 100_000:.2f} L"


def plot_month_vs_total(month_name: str, save: bool = True):
    """
    Plot a single month's sales trend alongside FY_Total across FY'23–FY'26.

    Parameters
    ----------
    month_name : str
        One of: 'June', 'July', 'August', 'September', 'October', 'November'
    save : bool
        If True, saves PNG to /mnt/user-data/outputs/
    """
    if month_name not in VALID_MONTHS:
        raise ValueError(f"month_name must be one of {VALID_MONTHS}, got '{month_name}'")

    fy_labels = ["FY'23", "FY'24", "FY'25", "FY'26"]
    x = np.arange(len(fy_labels))

    fig, ax = plt.subplots(figsize=(10, 6))
    fig.patch.set_facecolor('#f9f9f9')
    ax.set_facecolor('#f9f9f9')

    for series in ['FY_Total', month_name]:
        subset = df[df['MONTH_NAME'] == series].set_index('FINANCIAL_YEAR').reindex(fy_labels)
        y = subset['ACTUAL_SALES'].values

        lw    = 2.8 if series == 'FY_Total' else 2.2
        ls    = '--' if series == 'FY_Total' else '-'
        ms    = 9   if series == 'FY_Total' else 8
        zord  = 5   if series == 'FY_Total' else 3
        bold  = 'bold' if series == 'FY_Total' else 'normal'
        offset = 16 if series == 'FY_Total' else 11

        ax.plot(x, y,
                label=series,
                color=colors[series],
                linewidth=lw,
                linestyle=ls,
                marker=markers[series],
                markersize=ms,
                zorder=zord)

        for xi, yi in zip(x, y):
            if pd.isna(yi):
                continue
            ax.annotate(
                to_lakhs(yi),
                xy=(xi, yi),
                xytext=(0, offset),
                textcoords='offset points',
                ha='center',
                va='bottom',
                fontsize=9,
                color=colors[series],
                fontweight=bold
            )

    ax.set_xticks(x)
    ax.set_xticklabels(fy_labels, fontsize=12, fontweight='bold')
    ax.set_xlabel('Financial Year', fontsize=13, labelpad=10)
    ax.set_ylabel('Actual Sales (in Lakhs)', fontsize=13, labelpad=10)
    ax.set_title(
        f'{month_name} Sales vs FY Total : FY\'23 to FY\'26',
        fontsize=14, fontweight='bold', pad=16
    )

    ax.yaxis.set_major_formatter(
        ticker.FuncFormatter(lambda val, _: f"{val/100_000:.0f} L")
    )

    ax.set_xlim(-0.3, 3.3)
    ax.margins(y=0.22)
    ax.grid(axis='y', linestyle='--', alpha=0.4, color='grey')
    ax.spines[['top', 'right']].set_visible(False)

    ax.legend(
        title='Series',
        title_fontsize=10,
        fontsize=10,
        loc='upper left',
        framealpha=0.85,
        edgecolor='#cccccc'
    )

    plt.tight_layout()

    if save:
        out_path = f'/mnt/user-data/outputs/sales_{month_name.lower()}_vs_total.png'
        plt.savefig(out_path, dpi=180, bbox_inches='tight')
        print(f"Saved → {out_path}")

    plt.show()
    return fig


# ── Example usage ─────────────────────────────────────────────────────────────
if __name__ == '__main__':
    for m in ['June', 'July', 'August', 'September', 'October', 'November']:
        plot_month_vs_total(m)